# Code mẫu — CNN và ResNet

CNN ghép nhiều bộ lọc cục bộ lại để đưa ra một nhãn cho cả ảnh. Ta dùng một thí nghiệm nét đứng và nét ngang để theo dõi shape qua từng lớp và xem mạng học ra sao. Sau đó ta tách riêng residual block để hiểu vai trò của đường đi tắt.

## Ví dụ 1: Học phân biệt nét đứng và nét ngang

Mỗi ảnh xám 8×8 có một nét sáng, cộng thêm chút nhiễu. Nhãn 0 là nét đứng, nhãn 1 là nét ngang. Train và validation được sinh bằng hai seed khác nhau. Đây là dữ liệu tự tạo để kiểm quy trình, chưa nói gì về khả năng nhận chữ viết thật.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(52)

def make_lines(count, seed):
    rng = np.random.default_rng(seed)
    images = np.zeros((count, 1, 8, 8), dtype=np.float32)
    labels = np.arange(count) % 2
    for i, label in enumerate(labels):
        position = int(rng.integers(2, 5))
        if label == 0:
            images[i, 0, 1:7, position:position + 2] = 1
        else:
            images[i, 0, position:position + 2, 1:7] = 1
    images = np.clip(images + rng.normal(0, 0.04, images.shape), 0, 1)
    return torch.tensor(images, dtype=torch.float32), torch.tensor(labels)

train_x, train_y = make_lines(40, 1)
valid_x, valid_y = make_lines(20, 2)
print("Train:", tuple(train_x.shape), torch.bincount(train_y).tolist())
print("Một ảnh đứng, threshold 0.5:")
print((train_x[0, 0] >= 0.5).int().numpy())

Train: (40, 1, 8, 8) [20, 20]
Một ảnh đứng, threshold 0.5:
[[0 0 0 0 0 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 0]]


Mỗi lớp có 20 ảnh, nên một model luôn đoán cùng một lớp sẽ đạt accuracy 0.5. Đó là mốc để so. In ảnh đầu ra dưới dạng 0 và 1, ta thấy nhãn “đứng” đúng là đi với một nét đứng. Kiểm dữ liệu bằng mắt như vậy trước khi train giúp tránh những lỗi gán nhãn ngớ ngẩn.

Mạng gồm hai lớp conv có bốn kênh. Sau lớp conv đầu, max pooling thu ảnh từ 8×8 xuống 4×4. Cuối cùng, `AdaptiveAvgPool2d(1)` lấy trung bình mỗi kênh trên toàn ảnh, để còn bốn đặc trưng, rồi một lớp tuyến tính cho ra hai logit. Cross-entropy nhận thẳng hai logit này.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(4, 4, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(1), nn.Linear(4, 2),
)
shapes = []
batch = train_x[:2]
with torch.no_grad():
    for layer in model:
        batch = layer(batch)
        shapes.append(tuple(batch.shape))
print("Shape qua từng lớp:", shapes)
print("Tham số:", sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

def measure(x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        return (nn.functional.cross_entropy(logits, y).item(),
                (logits.argmax(1) == y).float().mean().item())

before = measure(valid_x, valid_y)
for step in range(70):
    model.train()
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(train_x), train_y)
    loss.backward()
    optimizer.step()
after = measure(valid_x, valid_y)
print("Valid loss/accuracy trước:", [round(v, 4) for v in before])
print("Valid loss/accuracy sau:", [round(v, 4) for v in after])
assert tuple(model(valid_x[:1]).shape) == (1, 2)
assert after[0] < before[0] and after[1] >= 0.9

Shape qua từng lớp: [(2, 4, 8, 8), (2, 4, 8, 8), (2, 4, 4, 4), (2, 4, 4, 4), (2, 4, 4, 4), (2, 4, 1, 1), (2, 4), (2, 2)]
Tham số: 198
Valid loss/accuracy trước: [0.7038, 0.5]
Valid loss/accuracy sau: [0.0, 1.0]


Đọc dòng shape từ trái sang phải. Conv đầu giữ 8×8 nhờ padding, và tạo bốn kênh. Pooling thu xuống 4×4. Pooling toàn ảnh còn 1×1. `Flatten(1)` bỏ hai trục không gian nhưng giữ trục batch, nên còn `(2, 4)`. Lớp cuối cho `(2, 2)`: hai ảnh, mỗi ảnh hai logit.

Mạng có 198 tham số. Conv đầu có $4\cdot9+4=40$ tham số. Conv sau có $4\cdot4\cdot9+4=148$. Lớp cuối có 10. Sau 70 bước, accuracy validation tăng từ 0.5 lên 1. Loss in ra 0.0 chỉ vì đã làm tròn tới bốn chữ số, chứ loss thật không bằng 0. Kết quả này chỉ đúng cho loại nét và mức nhiễu đã sinh ở trên.

## Ví dụ 2: Nhánh bằng 0 có giữ được cả giá trị âm không?

Một residual block tính `x + F(x)`. Khi nhánh F bằng 0, phép cộng giữ nguyên x. Nhưng nếu đặt ReLU sau phép cộng, phần âm vẫn bị cắt. Ta in cả đầu ra lẫn gradient để thấy riêng hai tác động này.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
branch = nn.Conv2d(1, 1, 1, bias=False)
with torch.no_grad():
    branch.weight.zero_()
x = torch.tensor([[[[2., -1.]]]], requires_grad=True)
before_relu = x + branch(x)
after_relu = torch.relu(before_relu)
after_relu.sum().backward()
print("Trước ReLU:", before_relu.detach().flatten().tolist())
print("Sau ReLU:", after_relu.detach().flatten().tolist())
print("Gradient theo x:", x.grad.flatten().tolist())
assert before_relu.detach().flatten().tolist() == [2.0, -1.0]
assert x.grad.flatten().tolist() == [1.0, 0.0]

Trước ReLU: [2.0, -1.0]
Sau ReLU: [2.0, 0.0]
Gradient theo x: [1.0, 0.0]


Trước ReLU, đầu ra đúng bằng đầu vào `[2, −1]`, vì nhánh F đóng góp 0. Sau ReLU, giá trị −1 thành 0. Gradient cũng vậy: tại giá trị 2, đường đi tắt cho gradient bằng 1, truyền thẳng về phía trước. Tại giá trị −1, ReLU chặn lại, gradient bằng 0.

Như vậy, một block có ReLU ở cuối không phải là phép giữ nguyên với mọi đầu vào, kể cả khi nhánh F bằng 0. Đường đi tắt giúp gradient đi qua dễ dàng, nhưng chỉ ở những chỗ mà ReLU cuối cho qua.

## Ví dụ 3: Đổi số kênh và thu nhỏ ảnh bằng conv 1×1

Block nhận đầu vào shape `(2, 4, 8, 8)`, nhưng nhánh chính trả ra `(2, 8, 4, 4)`: số kênh tăng gấp đôi, ảnh thu nhỏ một nửa. Muốn cộng được, nhánh đi tắt cũng phải ra đúng shape đó. Ta dùng một conv 1×1 với stride 2 trên nhánh đi tắt.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(523)
x = torch.ones(2, 4, 8, 8)
main = nn.Conv2d(4, 8, 3, stride=2, padding=1)
skip = nn.Conv2d(4, 8, 1, stride=2, bias=False)
a, b = main(x), skip(x)
result = torch.relu(a + b)
print("Nhánh chính:", tuple(a.shape))
print("Skip projection:", tuple(b.shape))
print("Kết quả:", tuple(result.shape))
print("Số ô input/output:", x.numel(), result.numel())
assert a.shape == b.shape == (2, 8, 4, 4)

Nhánh chính: (2, 8, 4, 4)
Skip projection: (2, 8, 4, 4)
Kết quả: (2, 8, 4, 4)
Số ô input/output: 512 256


Cả hai nhánh đều cho tám kênh trên lưới 4×4, nên cộng được với nhau. Đầu vào có 512 phần tử, đầu ra chỉ còn 256. Vì số phần tử khác nhau, không thể dùng `reshape` để ép đầu vào thành shape đầu ra. Và kể cả nếu ép được, `reshape` cũng không học được cách biến đổi bốn kênh thành tám kênh. Conv 1×1 trên nhánh đi tắt là một phép biến đổi có trọng số học được, làm đúng việc ấy.